In [1]:
import os
import re
import pandas as pd
import numpy as np
import pdfplumber
import spacy

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

nlp = spacy.load("en_core_web_sm")
stop_words = nlp.Defaults.stop_words

In [2]:
import os 
print(os.listdir(".."))
print(os.listdir("../data"))
print(os.listdir("../data/data"))

['.DS_Store', 'requirements.txt', 'README.md', 'venv', 'main.py', 'data', 'notebooks', 'src']
['.DS_Store', 'Resume', '.gitkeep', 'data']
['AGRICULTURE', 'ARTS', '.DS_Store', 'SALES', 'CONSULTANT', 'DIGITAL-MEDIA', 'CHEF', 'HEALTHCARE', 'PUBLIC-RELATIONS', 'AVIATION', 'BANKING', 'ACCOUNTANT', 'INFORMATION-TECHNOLOGY', 'HR', 'CONSTRUCTION', 'DESIGNER', 'FINANCE', 'FITNESS', 'BUSINESS-DEVELOPMENT', 'APPAREL', 'ADVOCATE', 'BPO', 'TEACHER', 'ENGINEERING', 'AUTOMOBILE']


In [ ]:
base_path = "../data/data"  

resumes = []
categories = []

for category in os.listdir(base_path):
    category_path = os.path.join(base_path, category)
    
    if os.path.isdir(category_path):
        for file in os.listdir(category_path):
            if file.endswith(".pdf"):
                file_path = os.path.join(category_path, file)
                
                try:
                    with pdfplumber.open(file_path) as pdf:
                        text = ""
                        for page in pdf.pages:
                            text += page.extract_text() or ""
                            
                        resumes.append(text)
                        categories.append(category)
                except:
                    continue

df = pd.DataFrame({
    "Category": categories,
    "Resume": resumes
})

df.head()
df.shape

(2484, 2)

In [4]:
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    
    return " ".join(tokens)
    

In [5]:
df["cleaned_resume"] = df["Resume"].apply(clean_text)

In [6]:
print(df.columns)

Index(['Category', 'Resume', 'cleaned_resume'], dtype='str')


In [7]:
job_description = """
Looking for a Data Scientist skilled in Python, Machine Learning,
Deep Learning, NLP, SQL, Pandas, Scikit-learn, TensorFlow,
data visualization, and model deployment.
"""

cleaned_jd = clean_text(job_description)

In [8]:
print(df.columns)

Index(['Category', 'Resume', 'cleaned_resume'], dtype='str')


In [9]:
vectorizer = TfidfVectorizer(max_features=5000)

all_text = df["cleaned_resume"].tolist()
all_text.append(cleaned_jd)

tfidf_matrix = vectorizer.fit_transform(all_text)

resume_vectors = tfidf_matrix[:-1]
jd_vector = tfidf_matrix[-1]

In [10]:
similarities = cosine_similarity(resume_vectors, jd_vector.reshape(1, -1))

df["similarity_score"] = similarities

scaler = MinMaxScaler()
df["score_percent"] = scaler.fit_transform(df[["similarity_score"]]) * 100

In [11]:
ranked_df = df.sort_values(by="score_percent", ascending=False)

ranked_df[["Category", "score_percent"]].head(10)

,Category,score_percent
337,CONSULTANT,100.000000
2371,ENGINEERING,83.334275
2471,AUTOMOBILE,77.267473
49,AGRICULTURE,60.368231
970,BANKING,56.444050
133,ARTS,48.551508
312,CONSULTANT,48.332625
16,AGRICULTURE,47.055282
424,DIGITAL-MEDIA,47.014966
1570,DESIGNER,45.904135
